In [1]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')
os.environ["HF_HOME"] = "/root/.cache/huggingface"

print("GPU 사용 가능 여부:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("연결된 GPU 이름:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU 사용 가능 여부: True
연결된 GPU 이름: Tesla T4


In [2]:
# 버전을 고정합니다. 상한이 없으면 어느 날 메이저 버전이 올라와 어제 돌던 코드가 멈춥니다
%pip install -q "transformers>=5.17,<6" "accelerate>=1.15,<2" "bitsandbytes>=0.50,<0.51"
%pip install -q "openai>=3.16,<4"

# 설치 후 pip이 의존성 충돌을 보고하면 [런타임] - [세션 다시 시작]을 한 번 누르고
# 이 셀 아래부터 다시 실행하세요

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 104.2 MB/s eta 0:00:00


In [3]:
import torch, json, re, time, os
from importlib.metadata import version, PackageNotFoundError

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("GPU가 없습니다. [런타임] - [런타임 유형 변경]에서 GPU를 선택하세요")


def pkgver(dist):
    """배포판 이름으로 설치된 버전을 읽습니다"""
    try:
        return version(dist)
    except PackageNotFoundError:
        return "미설치"


# 설치된 버전을 찍어둡니다. 나중에 결과가 이상하면 이 숫자부터 확인합니다
print("\n[설치된 버전]")
for d in ["transformers", "accelerate", "bitsandbytes", "openai"]:
    print(f"  {d:16s} {pkgver(d)}")

GPU : Tesla T4
VRAM : 15.6 GB

[설치된 버전]
  transformers     5.18.0
  accelerate       1.15.0
  bitsandbytes     0.50.2
  openai           3.24.0


In [4]:
BACKEND = "local"                       # "local" 또는 "endpoint"
ENDPOINT_URL = "http://localhost:8000/v1"
ENDPOINT_MODEL = "Qwen/Qwen2.5-7B-Instruct"
ENDPOINT_KEY = "dummy-key"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [5]:
# 토크나이저는 두 백엔드가 모두 씁니다. 분기 밖에서 임포트합니다.
# 안에 두면 "endpoint"로 바꾸는 순간 AutoTokenizer가 없다는 오류가 납니다
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if BACKEND == "local":
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    _model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb,
                                                 dtype=torch.float16,   # 양자화되지 않는 계층의 정밀도. T4는 bf16 미지원이라 명시합니다
                                                 device_map="auto")
    _model.eval()
    print(f"로컬 모델 로드 완료. VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    from openai import OpenAI
    _client = OpenAI(base_url=ENDPOINT_URL, api_key=ENDPOINT_KEY)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)   # 토큰 계산용으로만 사용
    print("엔드포인트 준비 완료")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

로컬 모델 로드 완료. VRAM 5.56 GB


In [6]:
def chat(messages, max_new_tokens=512, temperature=0.2):
    if BACKEND == "endpoint":
        r = _client.chat.completions.create(model=ENDPOINT_MODEL, messages=messages,
                                           temperature=temperature, max_tokens=max_new_tokens)
        return r.choices[0].message.content or ""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(text, return_tensors="pt").to(_model.device)
    # temperature를 do_sample=False와 함께 넘기면 호출마다 경고가 뜹니다.
    # 샘플링을 켤 때만 넘깁니다
    gen = {"max_new_tokens": max_new_tokens, "pad_token_id": tokenizer.eos_token_id}
    gen.update({"do_sample": True, "temperature": temperature} if temperature > 0
               else {"do_sample": False})
    with torch.no_grad():
        out = _model.generate(**ids, **gen)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

In [7]:
messages = [
    {"role": "system", "content": "당신은 사내 규정 안내 담당자입니다."},
    {"role": "user", "content": "연차 이월이 가능한가요?"},
    {"role": "assistant", "content": "원칙적으로는 불가능합니다."},
    {"role": "user", "content": "예외는 없나요?"},
]

rendered = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print(rendered)

<|im_start|>system
당신은 사내 규정 안내 담당자입니다.<|im_end|>
<|im_start|>user
연차 이월이 가능한가요?<|im_end|>
<|im_start|>assistant
원칙적으로는 불가능합니다.<|im_end|>
<|im_start|>user
예외는 없나요?<|im_end|>
<|im_start|>assistant



In [8]:
without_gen = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
print(repr(without_gen[-120:]))

'_start|>user\n연차 이월이 가능한가요?<|im_end|>\n<|im_start|>assistant\n원칙적으로는 불가능합니다.<|im_end|>\n<|im_start|>user\n예외는 없나요?<|im_end|>\n'


In [9]:
# 생성 프롬프트 없이 생성하면 어떻게 되는지 확인합니다
ids = tokenizer(without_gen, return_tensors="pt").to(_model.device)
with torch.no_grad():
    out = _model.generate(**ids, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

公司在某些情况下允许年假跨年度使用。具体规定可能因公司而异，但通常需要满足以下条件：
1. 公司政策允许年假跨年度使用。
2. 员工需要提前申请并获得批准。
3. 可能会有一些限制，比如不能跨两年使用等。

请查阅您公司的具体规定或咨询人力资源部门以获取准确信息。


In [10]:
# 모델마다 템플릿이 다릅니다. 두 모델을 비교해봅시다
# meta-llama는 게이트 저장소라 로그인이 필요합니다. 게이트 없는 미러를 먼저 시도합니다
from transformers import AutoTokenizer

tok_qwen = tokenizer
tok_llama = None
for repo in ["unsloth/Llama-3.2-1B-Instruct",        # 게이트 없음. 템플릿은 원본과 같습니다
             "meta-llama/Llama-3.2-1B-Instruct"]:    # 원본. 로그인이 되어 있으면 이쪽이 받아집니다
    try:
        tok_llama = AutoTokenizer.from_pretrained(repo)
        print(f"Llama 토크나이저 로드 : {repo}")
        break
    except Exception as e:
        print(f"건너뜀 : {repo} ({type(e).__name__})")

simple = [{"role": "system", "content": "You are helpful."},
          {"role": "user", "content": "Hi"}]

print("\n=== Qwen2.5 ===")
print(tok_qwen.apply_chat_template(simple, add_generation_prompt=True, tokenize=False))
print("\n=== Llama 3.2 ===")
if tok_llama is not None:
    print(tok_llama.apply_chat_template(simple, add_generation_prompt=True, tokenize=False))
else:
    print("(토크나이저를 받지 못했습니다. 아래 설명의 예시 출력을 보세요)")

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Llama 토크나이저 로드 : unsloth/Llama-3.2-1B-Instruct

=== Qwen2.5 ===
<|im_start|>system
You are helpful.<|im_end|>
<|im_start|>user
Hi<|im_end|>
<|im_start|>assistant


=== Llama 3.2 ===
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 06 Oct 2026

You are helpful.<|eot_id|><|start_header_id|>user<|end_header_id|>

Hi<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [11]:
%%writefile policy.md
## 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 한다. 시업 시각은 오전 10시, 종업 시각은 오후 7시로 하되, 유연근무제 적용자는 제4조에 따른다.

## 제4조 (유연근무제)
임직원은 오전 8시부터 오전 11시 사이에 시업 시각을 자율적으로 선택할 수 있다. 코어 타임은 오전 11시부터 오후 4시까지로 한다.

## 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

## 제8조 (경조 휴가)
본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 본인 및 배우자의 부모 사망 5일, 조부모 및 형제자매 사망 3일을 부여한다.

## 제10조 (사내 데이터의 외부 반출)
고객 개인정보, 미공개 재무 정보, 기술 설계 문서를 포함한 사내 데이터는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다.

Overwriting policy.md


In [12]:
with open("policy.md", encoding="utf-8") as f:
    doc = f.read()
chunks = [c.strip() for c in doc.split("## ") if c.strip()]
chunks = ["## " + c for c in chunks]

SYSTEM = """당신은 사내 규정 안내 담당자입니다.
제공된 규정에 근거해서만 답변하고, 근거 조항을 반드시 표기하세요.
규정에 없는 내용이면 확인되지 않는다고 답하세요.
답변은 반드시 한국어로만 작성하세요."""

TOOLS_DEF = json.dumps([
    {"name": "search_policy", "description": "규정을 검색합니다", "parameters": {"type": "object"}},
    {"name": "list_chapters", "description": "장 목록을 반환합니다", "parameters": {"type": "object"}},
], ensure_ascii=False)

question = "연차를 다음 해로 넘길 수 있나요?"
history = "이전 대화: 근무 시간을 물어봤고 오전 10시부터라고 답했습니다. " * 6


def ntok(s):
    return len(tokenizer.encode(s))


budget = {
    "시스템 프롬프트": ntok(SYSTEM),
    "도구 정의": ntok(TOOLS_DEF),
    f"검색 문서 (전체 {len(chunks)}개)": sum(ntok(c) for c in chunks),
    "대화 히스토리": ntok(history),
    "사용자 질문": ntok(question),
}
total = sum(budget.values())

print(f"{'구성 요소':<24} {'토큰':>8} {'비중':>8}")
print("-" * 44)
for k, v in budget.items():
    print(f"{k:<24} {v:>8} {v/total:>7.0%}")
print("-" * 44)
print(f"{'합계':<24} {total:>8}")

구성 요소                          토큰       비중
--------------------------------------------
시스템 프롬프트                       63      9%
도구 정의                          58      8%
검색 문서 (전체 6개)                 445     61%
대화 히스토리                       151     21%
사용자 질문                         13      2%
--------------------------------------------
합계                            730


In [13]:
# top-k를 줄였을 때 예산이 어떻게 변하는지 계산합니다
fixed = budget["시스템 프롬프트"] + budget["도구 정의"] + budget["사용자 질문"]

print(f"{'top-k':>6} {'문서 토큰':>10} {'전체 토큰':>10} {'절감률':>8}")
print("-" * 38)
for k in [len(chunks), 5, 3, 1]:
    doc_tokens = sum(ntok(c) for c in chunks[:k])
    tot = fixed + budget["대화 히스토리"] + doc_tokens
    print(f"{k:>6} {doc_tokens:>10} {tot:>10} {1 - tot/total:>7.0%}")

 top-k      문서 토큰      전체 토큰      절감률
--------------------------------------
     6        445        730      0%
     5        384        669      8%
     3        216        501     31%
     1         88        373     49%


In [14]:
# top-k별 답변을 비교합니다
def rag_answer(question, k):
    ctx = "\n\n".join(chunks[:k])
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}"}]
    return chat(msgs, max_new_tokens=250)


for k in [len(chunks), 3, 1]:
    print(f"=== top-{k} ===")
    print(rag_answer("연차를 다음 해로 넘길 수 있나요?", k)[:200])
    print()

=== top-6 ===
네, 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있습니다. 그러나 이월은 회사의 업무상 사유로 사용하지 못한 경우에 한하며, 이 때는 인사팀장의 승인을 받아야 합니다. 또한 이월 가능 일수는 최대 5일까지입니다. 이는 제7조에 근거합니다.

=== top-3 ===
해당 규정에는 연차를 다음 해로 넘길 수 있는지에 대한 내용이 명시되어 있지 않습니다. 따라서, 현재 제공된 규정에 따르면 확인된 답변은 '확인되지 않음'입니다.

=== top-1 ===
해당 규정에서는 연차를 다음 해로 넘길 수 있는지에 대한 내용이 명시되어 있지 않습니다. 따라서, 이 문제에 대한 답변을 제공하기 위해서는 추가적인 규정이나 회사의 정책이 필요합니다. 따라서, 현재 제공된 규정에 따르면 확인할 수 없는 사항입니다.



In [15]:
# GQA(그룹 질의 어텐션)는 여러 질의 헤드가 키·값 헤드 하나를 나눠 쓰는 구조이고 Qwen2.5가 이 구조입니다.
# 플래시 어텐션 커널이 없는 GPU(T4 등)에서는 GQA 경로가 느리고 메모리를 많이 쓰는 기본 커널로 떨어지므로 여기서만 끕니다.
# 키·값을 헤드 수만큼 복제해 메모리 효율 커널로 보냅니다. 계산 결과는 같습니다
import transformers.integrations.sdpa_attention as sdpa_mod

if torch.cuda.get_device_capability()[0] < 8:
    sdpa_mod.use_gqa_in_sdpa = lambda *args, **kwargs: False

In [16]:
# 채우기용 문서를 만듭니다. 정답 문서와 비슷한 주제(휴가 운영 세칙)이지만 정답은 들어 있지 않습니다
# 정답과 무관한 문서로 채우면 모델이 정답 문서를 쉽게 골라내 세 위치가 모두 100%로 나옵니다.
# 비슷한 문서 사이에 묻혀 있어야 위치에 따른 차이가 드러납니다
N_FILLER = 100          # 시간이 부족하면 60으로 줄이세요. 40 미만에서는 세 위치가 모두 100%로 나오기 쉽습니다

# 컨텍스트 윈도우 크기를 코드에 박지 않습니다. 모델이 알려주는 값을 씁니다.
# 모델을 바꾸면 이 숫자도 따라 바뀌어야 하는데, 박아두면 따라오지 않습니다
if BACKEND == "local":
    WINDOW = _model.config.max_position_embeddings
else:
    WINDOW = getattr(tokenizer, "model_max_length", 32768)
print(f"컨텍스트 윈도우 {WINDOW:,} 토큰 (모델 설정에서 읽었습니다)")

LEAVE_KINDS = ["병가", "경조 휴가", "공가", "보상 휴가", "장기근속 휴가", "교육 휴가", "출산 전후 휴가", "가족 돌봄 휴가"]
APPROVERS = ["부서장", "본부장", "대표이사", "총무팀장"]   # 정답 문서의 승인권자 '인사팀장'은 넣지 않습니다
DAYS = [2, 3, 7, 10]                                       # 정답 문서의 '5일'은 넣지 않습니다

filler = [
    f"## 부칙 제{i}항 ({LEAVE_KINDS[i % len(LEAVE_KINDS)]}의 운영)\n"
    f"{LEAVE_KINDS[i % len(LEAVE_KINDS)]}는 당해 연도 안에 사용함을 원칙으로 하되, "
    f"부득이한 사유가 있는 경우 {APPROVERS[i % len(APPROVERS)]}의 승인을 받아 "
    f"최대 {DAYS[i % len(DAYS)]}일까지 다음 분기 안에 나누어 사용할 수 있다. "
    f"신청은 사용 예정일 {i % 4 + 1}일 전까지 하며, 증빙 서류는 복귀 후 {i % 3 + 6}일 이내에 제출한다. "
    f"기한 안에 사용하지 않은 일수는 소멸하며 금전으로 보상하지 않는다."
    for i in range(1, N_FILLER + 1)
]

gold_doc = [c for c in chunks if "이월" in c][0]
filler_tok = sum(ntok(f) for f in filler)
print(f"정답 문서 : {gold_doc[:40]}...")
print(f"채우기 문서 {len(filler)}개, 총 {filler_tok:,} 토큰")
print(f"컨텍스트 윈도우 대비 {filler_tok/WINDOW:.0%} 사용  (컨텍스트 윈도우 {WINDOW:,} 토큰)")

컨텍스트 윈도우 32,768 토큰 (모델 설정에서 읽었습니다)
정답 문서 : ## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는...
채우기 문서 100개, 총 13,241 토큰
컨텍스트 윈도우 대비 40% 사용  (컨텍스트 윈도우 32,768 토큰)


In [17]:
# 정답 문서의 위치를 바꿔가며 정답률을 측정합니다
# 정답 판정은 "인사팀장"과 "5일"이 둘 다 답변에 있는지로 합니다. 둘 다 정답 문서에만 등장합니다
# 질문은 정답 문서의 표현("연차", "이월")을 그대로 쓰지 않습니다. 표현이 겹치면 모델이 글자만 맞춰 찾아내므로
# 위치에 따른 차이가 잘 안 보입니다
# 이 실험에서는 샘플링을 켭니다. 그리디(temperature=0)는 같은 입력에 항상 같은 답을 주므로
# 몇 번을 돌려도 0%와 100% 사이에 아무 값도 나오지 않습니다
PROBE_TEMP = 0.7
PROBE_TRIALS = 5
PROBE_QUESTION = "작년에 다 쓰지 못한 휴가를 올해 쓸 수 있는 조건과 승인권자는?"


def is_hit(ans):
    return "인사팀장" in ans and "5일" in ans


def probe(position, trials=PROBE_TRIALS):
    """position: 'front' | 'middle' | 'back'"""
    if position == "front":
        ctx = [gold_doc] + filler
    elif position == "middle":
        mid = len(filler) // 2
        ctx = filler[:mid] + [gold_doc] + filler[mid:]
    else:
        ctx = filler + [gold_doc]

    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) + f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        hits += int(is_hit(ans))
    return hits / trials


print(f"{'위치':>8} {'정답률':>8}   (각 {PROBE_TRIALS}회 시행)")
print("-" * 32)
for pos in ["front", "middle", "back"]:
    print(f"{pos:>8} {probe(pos):>7.0%}")

      위치      정답률   (각 5회 시행)
--------------------------------
   front      0%
  middle    100%
    back     80%


In [18]:
# 정답 문서의 위치를 바꿔가며 정답률을 측정합니다
# 정답 판정은 "인사팀장"과 "5일"이 둘 다 답변에 있는지로 합니다. 둘 다 정답 문서에만 등장합니다
# 질문은 정답 문서의 표현("연차", "이월")을 그대로 쓰지 않습니다. 표현이 겹치면 모델이 글자만 맞춰 찾아내므로
# 위치에 따른 차이가 잘 안 보입니다
# 이 실험에서는 샘플링을 켭니다. 그리디(temperature=0)는 같은 입력에 항상 같은 답을 주므로
# 몇 번을 돌려도 0%와 100% 사이에 아무 값도 나오지 않습니다
PROBE_TEMP = 0.7
PROBE_TRIALS = 5
PROBE_QUESTION = "작년에 다 쓰지 못한 휴가를 올해 쓸 수 있는 조건과 승인권자는?"


def is_hit(ans):
    return "인사팀장" in ans and "5일" in ans


def probe(position, trials=PROBE_TRIALS):
    """position: 'front' | 'middle' | 'back'"""
    if position == "front":
        ctx = [gold_doc] + filler
    elif position == "middle":
        mid = len(filler) // 2
        ctx = filler[:mid] + [gold_doc] + filler[mid:]
    else:
        ctx = filler + [gold_doc]

    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) + f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        hits += int(is_hit(ans))
    return hits / trials


print(f"{'위치':>8} {'정답률':>8}   (각 {PROBE_TRIALS}회 시행)")
print("-" * 32)
for pos in ["front", "middle", "back"]:
    print(f"{pos:>8} {probe(pos):>7.0%}")

      위치      정답률   (각 5회 시행)
--------------------------------
   front     40%
  middle    100%
    back     60%


In [19]:
# GQA(그룹 질의 어텐션)는 여러 질의 헤드가 키·값 헤드 하나를 나눠 쓰는 구조이고 Qwen2.5가 이 구조입니다.
# 플래시 어텐션 커널이 없는 GPU(T4 등)에서는 GQA 경로가 느리고 메모리를 많이 쓰는 기본 커널로 떨어지므로 여기서만 끕니다.
# 키·값을 헤드 수만큼 복제해 메모리 효율 커널로 보냅니다. 계산 결과는 같습니다
import transformers.integrations.sdpa_attention as sdpa_mod

if torch.cuda.get_device_capability()[0] < 8:
    sdpa_mod.use_gqa_in_sdpa = lambda *args, **kwargs: False

In [20]:
# 채우기용 문서를 만듭니다. 정답 문서와 비슷한 주제(휴가 운영 세칙)이지만 정답은 들어 있지 않습니다
# 정답과 무관한 문서로 채우면 모델이 정답 문서를 쉽게 골라내 세 위치가 모두 100%로 나옵니다.
# 비슷한 문서 사이에 묻혀 있어야 위치에 따른 차이가 드러납니다
N_FILLER = 100          # 시간이 부족하면 60으로 줄이세요. 40 미만에서는 세 위치가 모두 100%로 나오기 쉽습니다

# 컨텍스트 윈도우 크기를 코드에 박지 않습니다. 모델이 알려주는 값을 씁니다.
# 모델을 바꾸면 이 숫자도 따라 바뀌어야 하는데, 박아두면 따라오지 않습니다
if BACKEND == "local":
    WINDOW = _model.config.max_position_embeddings
else:
    WINDOW = getattr(tokenizer, "model_max_length", 32768)
print(f"컨텍스트 윈도우 {WINDOW:,} 토큰 (모델 설정에서 읽었습니다)")

LEAVE_KINDS = ["병가", "경조 휴가", "공가", "보상 휴가", "장기근속 휴가", "교육 휴가", "출산 전후 휴가", "가족 돌봄 휴가"]
APPROVERS = ["부서장", "본부장", "대표이사", "총무팀장"]   # 정답 문서의 승인권자 '인사팀장'은 넣지 않습니다
DAYS = [2, 3, 7, 10]                                       # 정답 문서의 '5일'은 넣지 않습니다

filler = [
    f"## 부칙 제{i}항 ({LEAVE_KINDS[i % len(LEAVE_KINDS)]}의 운영)\n"
    f"{LEAVE_KINDS[i % len(LEAVE_KINDS)]}는 당해 연도 안에 사용함을 원칙으로 하되, "
    f"부득이한 사유가 있는 경우 {APPROVERS[i % len(APPROVERS)]}의 승인을 받아 "
    f"최대 {DAYS[i % len(DAYS)]}일까지 다음 분기 안에 나누어 사용할 수 있다. "
    f"신청은 사용 예정일 {i % 4 + 1}일 전까지 하며, 증빙 서류는 복귀 후 {i % 3 + 6}일 이내에 제출한다. "
    f"기한 안에 사용하지 않은 일수는 소멸하며 금전으로 보상하지 않는다."
    for i in range(1, N_FILLER + 1)
]

gold_doc = [c for c in chunks if "이월" in c][0]
filler_tok = sum(ntok(f) for f in filler)
print(f"정답 문서 : {gold_doc[:40]}...")
print(f"채우기 문서 {len(filler)}개, 총 {filler_tok:,} 토큰")
print(f"컨텍스트 윈도우 대비 {filler_tok/WINDOW:.0%} 사용  (컨텍스트 윈도우 {WINDOW:,} 토큰)")

컨텍스트 윈도우 32,768 토큰 (모델 설정에서 읽었습니다)
정답 문서 : ## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는...
채우기 문서 100개, 총 13,241 토큰
컨텍스트 윈도우 대비 40% 사용  (컨텍스트 윈도우 32,768 토큰)


In [21]:
# 정답 문서의 위치를 바꿔가며 정답률을 측정합니다
# 정답 판정은 "인사팀장"과 "5일"이 둘 다 답변에 있는지로 합니다. 둘 다 정답 문서에만 등장합니다
# 질문은 정답 문서의 표현("연차", "이월")을 그대로 쓰지 않습니다. 표현이 겹치면 모델이 글자만 맞춰 찾아내므로
# 위치에 따른 차이가 잘 안 보입니다
# 이 실험에서는 샘플링을 켭니다. 그리디(temperature=0)는 같은 입력에 항상 같은 답을 주므로
# 몇 번을 돌려도 0%와 100% 사이에 아무 값도 나오지 않습니다
PROBE_TEMP = 0.7
PROBE_TRIALS = 5
PROBE_QUESTION = "작년에 다 쓰지 못한 휴가를 올해 쓸 수 있는 조건과 승인권자는?"


def is_hit(ans):
    return "인사팀장" in ans and "5일" in ans


def probe(position, trials=PROBE_TRIALS):
    """position: 'front' | 'middle' | 'back'"""
    if position == "front":
        ctx = [gold_doc] + filler
    elif position == "middle":
        mid = len(filler) // 2
        ctx = filler[:mid] + [gold_doc] + filler[mid:]
    else:
        ctx = filler + [gold_doc]

    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) + f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        hits += int(is_hit(ans))
    return hits / trials


print(f"{'위치':>8} {'정답률':>8}   (각 {PROBE_TRIALS}회 시행)")
print("-" * 32)
for pos in ["front", "middle", "back"]:
    print(f"{pos:>8} {probe(pos):>7.0%}")

      위치      정답률   (각 5회 시행)
--------------------------------
   front      0%
  middle    100%
    back     80%


In [22]:
# 지시문을 끝에 반복하면 중간 소실이 완화되는지 확인합니다
def probe_with_reminder(trials=PROBE_TRIALS):
    mid = len(filler) // 2
    ctx = filler[:mid] + [gold_doc] + filler[mid:]
    hits = 0
    for _ in range(trials):
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user",
                 "content": "[규정]\n" + "\n\n".join(ctx) +
                            "\n\n위 규정 전체를 빠짐없이 검토하세요. 특히 쓰지 못한 휴가를 다음 해에 쓰는 조항을 찾으세요."
                            f"\n\n질문: {PROBE_QUESTION}"}]
        ans = chat(msgs, max_new_tokens=200, temperature=PROBE_TEMP)
        print(f"실제 답변: {ans}")
        hits += int(is_hit(ans))
    return hits / trials


print(f"중간 배치 (지시 반복 없음) : {probe('middle'):.0%}")
print(f"중간 배치 (지시 반복 있음) : {probe_with_reminder():.0%}")

중간 배치 (지시 반복 없음) : 100%
실제 답변: 작년에 다 쓰지 못한 휴가를 올해에 사용할 수 있는 조건과 승인권자는 다음과 같습니다:

- **경조 휴가**: 
  - 조건: 부득이한 사유가 있는 경우 본부장의 승인을 받아 최대 3일까지 다음 분기 안에 나누어 사용할 수 있다.
  - 승인권자: 본부장

- **공가**: 
  - 조건: 부득이한 사유가 있는 경우 대표이사의 승인을 받아 최대 7일까지 다음 분기 안에 나누어 사용할 수 있다.
  - 승인권자: 대표이사

- **보상 휴가**: 
  - 조건: 부득이한 사유가 있는 경우 총무팀장의 승인을 받아 최대 1
실제 답변: 작년에 다 쓰지 못한 휴가를 올해에 이월하여 사용하는 조건과 승인권자가 명시된 조항은 없습니다. 

특히, "제7조 (연차 휴가의 이월)" 항목에서 이와 관련된 내용이 있지만, 해당 항목에서는 다음 연도로 이월할 수 없다는 원칙이 제시되어 있으며, 단적으로 "다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다"고 명시되어 있습니다.

따라서, 작년에 다 쓰지 못한 휴가는 원칙적으로 올해 사용할 수 없습니다.
실제 답변: 작년에 사용하지 못한 휴가를 올해에 이월하여 사용하는 조항은 없습니다. 모든 휴가 유형에 대해 "당해 연도에 사용함을 원칙으로 한다"고 명시되어 있으며, 이월 조건은 없습니다. 

특히 제7조에서는 "당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다"고 명시되어 있습니다. 이외의 휴가 유형들에서도 비슷한 조항이 있으며, 모두 당해 연도 내에서 사용하도록 정해져 있습니다.

따라서 작년에 남은 휴가를 올해에 사용할 수 있는 조건이나 승인권자는 없습니다.
실제 답변: 작년에 다 쓰지 못한 휴가를 올해 쓸 수 있는 조건과 승인권자는 다음과 같습니다:

- **경조 휴가**: 당해 연도에 사용하지 못한 경조 휴가는 다음 연도 1분기 내로 이월하여 

In [23]:
# 가이드 : 규약 문서
GUIDE = """[사내 답변 작성 규약]
1. 답변은 3문장 이내로 작성한다.
2. 반드시 근거 조항 번호를 '제N조' 형식으로 표기한다. 단, 규정에서 확인되지 않는 경우에는 표기하지 않는다.
3. 조건이 있는 규정은 조건을 빠뜨리지 않는다.
4. 추측이나 일반론을 쓰지 않는다. 규정에 없으면 '규정에서 확인되지 않습니다'라고 답한다.
5. 존댓말을 사용한다."""

In [24]:
# 센서 : 규약 준수를 기계적으로 검사합니다. 모델이 아니라 규칙 기반입니다
# 용어 대응 : 여기서 말하는 '규칙 기반 센서'가 이론 레슨 4.5절의 '계산형 센서'입니다.
#            뒤의 트랙 2에 나오는 '판정 모델 센서'가 이론 레슨의 '추론형 센서'입니다
def check_answer(ans: str):
    issues = []
    sentences = [s for s in re.split(r"[.!?]\s*", ans.strip()) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장 (3문장 이내여야 함)")
    # 규약4에 따라 "확인되지 않습니다"로 답한 경우에는 인용할 조항이 없습니다.
    # 이 예외를 빼먹으면 규정에 없는 질문은 몇 번을 다시 써도 통과하지 못합니다
    unanswerable = bool(re.search(r"확인되지\s?않", ans))
    if not unanswerable and not re.search(r"제\s?\d+\s?조", ans):
        issues.append("규약2 위반: 근거 조항 번호 없음")
    # 존댓말은 '~니다'와 '~세요/~시오'로 봅니다. '습니다'와 '입니다'만 찾으면
    # '가능합니다', '바랍니다', '주십시오'가 전부 반말로 잡힙니다
    if not re.search(r"(니다|세요|시오)", ans):
        issues.append("규약5 위반: 존댓말 아님")
    return {"pass": len(issues) == 0, "issues": issues}


# 센서 동작 확인
print(check_answer("연차는 이월할 수 없다."))
print(check_answer("원칙적으로 이월이 불가능합니다. 다만 업무상 사유가 있으면 인사팀장 승인으로 최대 5일까지 가능합니다. 근거는 제7조입니다."))
print(check_answer("해당 내용은 규정에서 확인되지 않습니다."))   # 조항 인용 면제가 동작하는지

{'pass': False, 'issues': ['규약2 위반: 근거 조항 번호 없음', '규약5 위반: 존댓말 아님']}
{'pass': True, 'issues': []}
{'pass': True, 'issues': []}


In [25]:
# 예외를 빼먹은 센서. 규약2를 무조건 요구합니다
def check_answer_buggy(ans: str):
    issues = []
    sentences = [s for s in re.split(r"[.!?]\s*", ans.strip()) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장")
    if not re.search(r"제\s?\d+\s?조", ans):          # 예외 없음
        issues.append("규약2 위반: 근거 조항 번호 없음")
    if not re.search(r"(니다|세요|시오)", ans):
        issues.append("규약5 위반: 존댓말 아님")
    return {"pass": len(issues) == 0, "issues": issues}


# 규약을 완벽하게 지킨 답변 다섯 개입니다. 사람이 채점하면 5/5입니다
perfect = [
    "원칙적으로 이월할 수 없습니다. 업무상 사유가 있으면 인사팀장 승인으로 5일까지 가능합니다. 근거는 제7조입니다.",
    "재택근무는 근무일 전일 오후 6시까지 신청해야 합니다. 근거는 제5조입니다.",
    "배우자 출산 휴가는 10일입니다. 근거는 제8조입니다.",
    "코어 타임은 오전 11시부터입니다. 근거는 제4조입니다.",
    "해당 내용은 규정에서 확인되지 않습니다.",           # 규약4를 지킨 답변
]

for name, sensor in [("예외 있음 (고친 센서)", check_answer),
                     ("예외 없음 (버그 센서)", check_answer_buggy)]:
    hit = sum(sensor(a)["pass"] for a in perfect)
    print(f"{name:<22} {hit}/{len(perfect)}  (상한 {hit/len(perfect):.0%})")

예외 있음 (고친 센서)          5/5  (상한 100%)
예외 없음 (버그 센서)          4/5  (상한 80%)


In [26]:
# 평가 세트
qa_set = [
    ("연차를 다음 해로 넘길 수 있나요?", "이월"),
    ("재택근무는 언제까지 신청해야 하나요?", "전일"),
    ("배우자가 출산하면 휴가가 며칠인가요?", "10"),
    ("코어 타임은 몇 시부터인가요?", "11"),
    ("점심시간은 몇 시인가요?", None),          # 규정에 없는 질문
]


def answer_once(q, guide=None):
    ctx = "\n\n".join(chunks)
    sys = SYSTEM + (f"\n\n{guide}" if guide else "")
    msgs = [{"role": "system", "content": sys},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    # 규약 준수 측정은 그리디로 합니다. 같은 레슨을 돌린 사람끼리 결과가 같아야
    # 조건 A·B·C의 차이가 하네스 때문인지 우연 때문인지 구분할 수 있습니다
    return chat(msgs, max_new_tokens=250, temperature=0.0)

In [27]:
# 조건 A : 가이드도 센서도 없음
print("=== A. 가이드 없음, 센서 없음 ===")
pass_a = 0
for q, _ in qa_set:
    a = answer_once(q)
    v = check_answer(a)
    pass_a += v["pass"]
    print(f"  [{'O' if v['pass'] else 'X'}] {q}")
    if not v["pass"]:
        print(f"       {v['issues']}")
print(f"규약 준수율 : {pass_a}/{len(qa_set)}")

=== A. 가이드 없음, 센서 없음 ===
  [X] 연차를 다음 해로 넘길 수 있나요?
       ['규약1 위반: 4문장 (3문장 이내여야 함)']
  [O] 재택근무는 언제까지 신청해야 하나요?
  [O] 배우자가 출산하면 휴가가 며칠인가요?
  [O] 코어 타임은 몇 시부터인가요?
  [X] 점심시간은 몇 시인가요?
       ['규약2 위반: 근거 조항 번호 없음']
규약 준수율 : 3/5


In [28]:
# 조건 B : 가이드만 제공
print("=== B. 가이드 있음, 센서 없음 ===")
pass_b = 0
for q, _ in qa_set:
    a = answer_once(q, guide=GUIDE)
    v = check_answer(a)
    pass_b += v["pass"]
    print(f"  [{'O' if v['pass'] else 'X'}] {q}")
print(f"규약 준수율 : {pass_b}/{len(qa_set)}")

=== B. 가이드 있음, 센서 없음 ===
  [O] 연차를 다음 해로 넘길 수 있나요?
  [O] 재택근무는 언제까지 신청해야 하나요?
  [O] 배우자가 출산하면 휴가가 며칠인가요?
  [O] 코어 타임은 몇 시부터인가요?
  [O] 점심시간은 몇 시인가요?
규약 준수율 : 5/5


In [29]:
# 실습 : 조건 C를 구현하세요. 가이드 + 센서 + 피드백 재주입
# 센서가 위반을 잡으면 위반 내용을 알려주고 다시 쓰게 합니다. 최대 3회.

def answer_with_sensor(q, max_retry=3):
    ctx = "\n\n".join(chunks)
    sys = SYSTEM + "\n\n" + GUIDE
    msgs = [{"role": "system", "content": sys},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]

    for attempt in range(max_retry):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = check_answer(ans)
        if v["pass"]:
            return ans, True, attempt + 1

        # 피드백 재주입 : 위반 내용을 구체적으로 알려줍니다
        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user",
                     "content": "작성한 답변이 규약을 위반했습니다.\n"
                                + "\n".join(f"- {i}" for i in v["issues"])
                                + "\n위반 사항을 고쳐 답변을 다시 작성하세요."})

    return ans, False, max_retry

In [30]:
print("=== C. 가이드 + 센서 + 피드백 재주입 ===")
pass_c, total_attempts = 0, 0
for q, _ in qa_set:
    a, ok, tries = answer_with_sensor(q)
    pass_c += ok
    total_attempts += tries
    print(f"  [{'O' if ok else 'X'}] {q}  (시도 {tries}회)")
print(f"규약 준수율 : {pass_c}/{len(qa_set)}")
print(f"평균 시도 횟수 : {total_attempts/len(qa_set):.1f}")

=== C. 가이드 + 센서 + 피드백 재주입 ===
  [O] 연차를 다음 해로 넘길 수 있나요?  (시도 1회)
  [O] 재택근무는 언제까지 신청해야 하나요?  (시도 1회)
  [O] 배우자가 출산하면 휴가가 며칠인가요?  (시도 1회)
  [O] 코어 타임은 몇 시부터인가요?  (시도 1회)
  [O] 점심시간은 몇 시인가요?  (시도 1회)
규약 준수율 : 5/5
평균 시도 횟수 : 1.0


In [31]:
print(f"{'조건':<28} {'준수율':>8}")
print("-" * 40)
print(f"{'A. 가이드 없음, 센서 없음':<28} {pass_a/len(qa_set):>7.0%}")
print(f"{'B. 가이드만':<28} {pass_b/len(qa_set):>7.0%}")
print(f"{'C. 가이드 + 센서':<28} {pass_c/len(qa_set):>7.0%}")

조건                                준수율
----------------------------------------
A. 가이드 없음, 센서 없음                 60%
B. 가이드만                         100%
C. 가이드 + 센서                     100%


In [32]:
STRICT_GUIDE = GUIDE + """
6. 답변의 첫 문장은 반드시 '규정에 따르면'으로 시작한다.
7. 숫자는 아라비아 숫자로만 표기한다. (예: '다섯 일'이 아니라 '5일')
8. 답변 끝에 '(근거: 제N조)' 형식으로 마무리한다. 규정에서 확인되지 않는 경우에는 생략한다."""


TAIL = r"\(근거\s?:\s?제\s?\d+\s?조\)"


def check_strict(ans: str):
    issues = []
    # check_answer와 같은 예외를 여기에도 둡니다. 센서를 강화할 때
    # 앞에서 고친 예외를 빼먹는 것이 실제로 가장 잦은 실수입니다
    unanswerable = bool(re.search(r"확인되지\s?않", ans))
    # 규약8이 요구하는 '(근거: 제N조)'는 문장이 아닙니다. 빼고 셉니다.
    # 빼지 않으면 규약1(3문장 이내)과 규약8이 서로를 위반하게 만듭니다
    body = re.sub(TAIL + r"\s*$", "", ans.strip())
    sentences = [s for s in re.split(r"[.!?]\s*", body) if s]
    if len(sentences) > 3:
        issues.append(f"규약1 위반: {len(sentences)}문장")
    if not unanswerable and not re.search(r"제\s?\d+\s?조", ans):
        issues.append("규약2 위반: 조항 번호 없음")
    if not ans.strip().startswith("규정에 따르면"):
        issues.append("규약6 위반: 첫 문장이 '규정에 따르면'으로 시작하지 않음")
    if re.search(r"(하나|둘|셋|넷|다섯|열)\s?일", ans):
        issues.append("규약7 위반: 숫자를 한글로 표기")
    # '끝에'라고 했으므로 끝에서 찾습니다. 아무 데나 있어도 통과시키면
    # 규약8이 사실상 규약2와 같은 검사가 됩니다
    if not unanswerable and not re.search(TAIL + r"\s*$", ans.strip()):
        issues.append("규약8 위반: 끝에 '(근거: 제N조)' 형식 없음")
    return {"pass": len(issues) == 0, "issues": issues}

In [33]:
# 반복 횟수별 통과율을 측정해 수렴 곡선을 만듭니다
# 각 질문에 대해 최대 n회까지 루프를 돌렸을 때의 통과율을 기록합니다

def loop_until_pass(q, max_iter, checker, guide):
    ctx = "\n\n".join(chunks)
    msgs = [{"role": "system", "content": SYSTEM + "\n\n" + guide},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    for i in range(max_iter):
        # 피드백이 메시지에 쌓이므로 입력이 매 회 달라집니다. 그리디여도 답이 바뀝니다
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = checker(ans)
        if v["pass"]:
            return True, i + 1
        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])
                                                + "\n고쳐서 다시 작성하세요."})
    return False, max_iter


print(f"{'최대 반복':>8} {'통과율':>8} {'평균 시도':>10}")
print("-" * 30)
for max_iter in [1, 2, 3, 5]:
    results = [loop_until_pass(q, max_iter, check_strict, STRICT_GUIDE) for q, _ in qa_set]
    rate = sum(r[0] for r in results) / len(results)
    avg = sum(r[1] for r in results) / len(results)
    print(f"{max_iter:>8} {rate:>7.0%} {avg:>10.1f}")

   최대 반복      통과율      평균 시도
------------------------------
       1     80%        1.0
       2    100%        1.2
       3    100%        1.2
       5    100%        1.2


In [34]:
# 실습 : loop_until_pass에 정체 종료를 추가하세요
# 요구사항
#  1) 직전 회차의 위반 목록과 이번 회차의 위반 목록이 같으면 즉시 중단
#  2) 종료 이유를 "성공" / "정체" / "한계" 중 하나의 문자열로 함께 반환
#     (정체로 멈춘 것은 사람에게 넘길 목록이 됩니다. 이것이 네 번째 조건인 위임의 출구입니다)
# 힌트 : 위반 목록은 문자열 리스트입니다. 리스트끼리 == 로 비교하면 됩니다

def loop_with_stall(q, max_iter, checker, guide):
    ctx = "\n\n".join(chunks)
    msgs = [{"role": "system", "content": SYSTEM + "\n\n" + guide},
            {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {q}"}]
    prev_issues = None

    for i in range(max_iter):
        ans = chat(msgs, max_new_tokens=250, temperature=0.0)
        v = checker(ans)
        if v["pass"]:
            return True, i + 1, "성공"

        # 정체 : 구체적으로 알려줬는데 같은 위반이 그대로 돌아왔습니다
        if v["issues"] == prev_issues:
            return False, i + 1, "정체"
        prev_issues = v["issues"]

        msgs.append({"role": "assistant", "content": ans})
        msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])
                                                + "\n고쳐서 다시 작성하세요."})

    return False, max_iter, "한계"

In [35]:
# 두 루프를 같은 조건에서 비교합니다. 통과율은 유지하면서 호출을 줄였는지 봅니다
from collections import Counter

MAX_ITER = 5
base = [loop_until_pass(q, MAX_ITER, check_strict, STRICT_GUIDE) for q, _ in qa_set]
stall = [loop_with_stall(q, MAX_ITER, check_strict, STRICT_GUIDE) for q, _ in qa_set]

print(f"{'루프':<22} {'통과율':>8} {'총 호출':>8}")
print("-" * 42)
print(f"{'정체 종료 없음':<22} {sum(r[0] for r in base)/len(base):>7.0%} {sum(r[1] for r in base):>8}")
print(f"{'정체 종료 있음':<22} {sum(r[0] for r in stall)/len(stall):>7.0%} {sum(r[1] for r in stall):>8}")

print("\n[종료 이유별 건수]")
for reason, n in Counter(r[2] for r in stall).items():
    print(f"  {reason} {n}건")

# 정체로 걸러진 질문은 버릴 로그가 아니라 고칠 작업 목록입니다
print("\n[정체로 멈춘 질문 = 사람에게 넘길 목록]")
for (q, _), r in zip(qa_set, stall):
    if r[2] == "정체":
        print(f"  {q}")

루프                          통과율     총 호출
------------------------------------------
정체 종료 없음                  100%        6
정체 종료 있음                  100%        6

[종료 이유별 건수]
  성공 5건

[정체로 멈춘 질문 = 사람에게 넘길 목록]


In [36]:
# 재시도가 반복되면 히스토리가 누적됩니다
ctx = "\n\n".join(chunks)
msgs = [{"role": "system", "content": SYSTEM + "\n\n" + STRICT_GUIDE},
        {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: 배우자 출산 휴가는 며칠인가요?"}]

for i in range(4):
    ans = chat(msgs, max_new_tokens=250, temperature=0.5)
    v = check_strict(ans)
    msgs.append({"role": "assistant", "content": ans})
    msgs.append({"role": "user", "content": "규약 위반:\n" + "\n".join(v["issues"])})
    tok = ntok(tokenizer.apply_chat_template(msgs, tokenize=False))
    print(f"{i+1}회차 후 : 메시지 {len(msgs)}개, 누적 {tok} 토큰")

1회차 후 : 메시지 4개, 누적 791 토큰
2회차 후 : 메시지 6개, 누적 868 토큰
3회차 후 : 메시지 8개, 누적 943 토큰
4회차 후 : 메시지 10개, 누적 990 토큰


In [37]:
# 실습 : 컴팩션을 구현하세요
# 실패한 시도의 전문을 모두 남기는 대신, 위반 항목만 누적 목록으로 압축합니다
# 힌트 : 히스토리를 버리고 "지금까지 지적된 위반 목록"만 유지합니다

def loop_with_compaction(q, max_iter=4):
    ctx = "\n\n".join(chunks)
    base = [{"role": "system", "content": SYSTEM + "\n\n" + STRICT_GUIDE}]
    known_issues = []

    for i in range(max_iter):
        # 여기에 압축된 메시지 구성과 루프를 작성하세요
        pass

    return None

In [38]:
print(loop_with_compaction("배우자 출산 휴가는 며칠인가요?"))

None


In [39]:
# 실습 : 진행 상황을 파일에 기록하며 여러 질문을 순차 처리하는 루프를 만드세요
# 요구사항
#  1) 각 질문 처리 후 결과를 progress.json에 기록
#  2) 다음 질문 처리 시 이전 결과 요약만 컨텍스트에 넣기
#  3) 중간에 커널이 죽어도 파일을 읽어 이어갈 수 있는 구조

def load_notes(note_file):
    """기록 파일을 읽습니다. 깨져 있으면 없는 것으로 칩니다.
    중단된 시점에 파일이 반쯤 쓰였을 수 있어, 여기서 막히면 재시작 자체가 안 됩니다"""
    if not os.path.exists(note_file):
        return {}
    try:
        with open(note_file, encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        print("기록 파일이 깨져 있어 처음부터 시작합니다")
        return {}


def process_with_notes(questions, note_file="progress.json"):
    notes = load_notes(note_file)

    for q in questions:
        if q in notes:
            print(f"[건너뜀] {q}")
            continue

        # 이전 결과는 요약만 컨텍스트에 넣습니다
        prev = "\n".join(f"- {k}: {v['summary']}" for k, v in list(notes.items())[-3:])
        note = f"\n\n[이전에 답한 항목]\n{prev}" if prev else ""

        ctx = "\n\n".join(chunks)
        msgs = [{"role": "system", "content": SYSTEM + "\n\n" + GUIDE},
                {"role": "user", "content": f"[규정]\n{ctx}{note}\n\n질문: {q}"}]
        ans = chat(msgs, max_new_tokens=250)

        notes[q] = {"answer": ans, "summary": ans[:60], "pass": check_answer(ans)["pass"]}
        with open(note_file, "w", encoding="utf-8") as f:
            json.dump(notes, f, ensure_ascii=False, indent=2)
        print(f"[완료] {q}")

    return notes

In [40]:
# 이 셀을 통째로 다시 실행해도 결과는 같습니다. 기록이 파일에 남아 있기 때문입니다
qs = [q for q, _ in qa_set]

print("=== 1회차 ===")
process_with_notes(qs)

print("\n=== 2회차 (커널이 죽었다 살아난 셈) ===")
process_with_notes(qs)

=== 1회차 ===
[완료] 연차를 다음 해로 넘길 수 있나요?
[완료] 재택근무는 언제까지 신청해야 하나요?
[완료] 배우자가 출산하면 휴가가 며칠인가요?
[완료] 코어 타임은 몇 시부터인가요?
[완료] 점심시간은 몇 시인가요?

=== 2회차 (커널이 죽었다 살아난 셈) ===
[건너뜀] 연차를 다음 해로 넘길 수 있나요?
[건너뜀] 재택근무는 언제까지 신청해야 하나요?
[건너뜀] 배우자가 출산하면 휴가가 며칠인가요?
[건너뜀] 코어 타임은 몇 시부터인가요?
[건너뜀] 점심시간은 몇 시인가요?


{'연차를 다음 해로 넘길 수 있나요?': {'answer': '네, 가능합니다. 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있으며, 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다. (제7조)',
  'summary': '네, 가능합니다. 당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 있으며, 회사의 업무상 사유',
  'pass': True},
 '재택근무는 언제까지 신청해야 하나요?': {'answer': '재택근무는 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다. 제5조에 따릅니다.',
  'summary': '재택근무는 근무일 전일 오후 6시까지 부서장의 승인을 받아야 합니다. 제5조에 따릅니다.',
  'pass': True},
 '배우자가 출산하면 휴가가 며칠인가요?': {'answer': '배우자가 출산하면 휴가가 10일간 부여됩니다. 제8조에 따릅니다.',
  'summary': '배우자가 출산하면 휴가가 10일간 부여됩니다. 제8조에 따릅니다.',
  'pass': True},
 '코어 타임은 몇 시부터인가요?': {'answer': '코어 타임은 오전 11시부터 오후 4시까지입니다. 제4조에 따릅니다.',
  'summary': '코어 타임은 오전 11시부터 오후 4시까지입니다. 제4조에 따릅니다.',
  'pass': True},
 '점심시간은 몇 시인가요?': {'answer': '규정에서 확인되지 않습니다.',
  'summary': '규정에서 확인되지 않습니다.',
  'pass': True}}

In [41]:
JUDGE_SYSTEM = """당신은 답변을 검증하는 판정자입니다.
규정 원문과 답변을 비교해 아래를 판정하고 JSON으로만 답하세요.
1. 답변 내용이 규정과 일치하는가 (사실 정확성)
2. 조건이나 예외를 빠뜨리지 않았는가
형식 : {"pass": true/false, "issues": ["문제점"]}
issues 항목은 한국어로 작성하고, JSON 외의 텍스트는 출력하지 마세요."""


def judge_sensor(question, answer):
    ctx = "\n\n".join(chunks)
    r = chat([{"role": "system", "content": JUDGE_SYSTEM},
              {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}\n답변: {answer}"}],
             max_new_tokens=300, temperature=0.0)
    m = re.search(r"\{.*\}", r, re.S)
    # 판정 자체가 실패했을 때 무엇을 돌려줄지를 정해두어야 합니다.
    #   실패하면 통과시킨다 : pass=True. 편하지만 그 순간부터 센서가 조용히 꺼집니다
    #   실패하면 막는다     : pass=False. 시끄럽지만 못 본 채 나가는 답변이 없습니다
    # 형식이 깨진 판정은 통과가 아니라 실패로 둡니다
    try:
        return json.loads(m.group(0)) if m else {"pass": False, "issues": ["판정 형식 오류: JSON을 찾지 못함"]}
    except json.JSONDecodeError:
        return {"pass": False, "issues": ["판정 형식 오류: JSON 파싱 실패"]}

In [42]:
# 실습 : 두 센서를 비교하세요
# 의도적으로 형식은 맞지만 내용이 틀린 답변, 내용은 맞지만 형식이 틀린 답변을 만들어
# 각 센서가 무엇을 잡고 무엇을 놓치는지 확인하세요

test_answers = [
    # (설명, 질문, 답변)
    ("형식 O, 내용 O", "연차 이월?",
     "규정에 따르면 원칙적으로 이월이 불가능합니다. 업무상 사유가 있으면 인사팀장 승인으로 최대 5일까지 가능합니다. (근거: 제7조)"),
    ("형식 O, 내용 X", "연차 이월?",
     "규정에 따르면 연차는 최대 10일까지 자유롭게 이월할 수 있습니다. 별도 승인은 필요하지 않습니다. (근거: 제7조)"),
    ("형식 X, 내용 O", "연차 이월?",
     "원칙적으로 불가능하지만 업무상 사유면 인사팀장 승인으로 5일까지 이월 가능"),
]

print(f"{'케이스':<16} {'규칙 센서':>10} {'판정 센서':>10}")
print("-" * 40)
for desc, q, a in test_answers:
    rule = check_answer(a)["pass"]
    llm = judge_sensor(q, a)["pass"]
    print(f"{desc:<16} {'통과' if rule else '탈락':>10} {'통과' if llm else '탈락':>10}")

케이스                   규칙 센서      판정 센서
----------------------------------------
형식 O, 내용 O               통과         통과
형식 O, 내용 X               통과         탈락
형식 X, 내용 O               탈락         통과
